In [ ]:
"""
ERA5 Temperature 95th Percentile Calculator

Core functions to compute day-of-year 95th percentile thresholds using a 
31-day centered sliding window (±15 days) across multiple years.
"""

import numpy as np
import pandas as pd
import xarray as xr
from typing import Tuple, Union


def calculate_tmax_p95(
    tmax_series: np.ndarray,
    dates: Union[pd.DatetimeIndex, pd.Series]
) -> np.ndarray:
    """
    Compute daily 95th percentile thresholds using a 31-day sliding window.
    
    For each calendar day (e.g., Jan 16), aggregates temperature values from
    ±15 days across all years (e.g., Jan 1–31 across all years), then computes
    the 95th percentile of that aggregated window.
    
    Parameters
    ----------
    tmax_series : np.ndarray
        1D array of daily maximum temperatures (length = number of days)
    dates : pd.DatetimeIndex or pd.Series
        Corresponding dates for each temperature value
    
    Returns
    -------
    np.ndarray
        Array of 95th percentile values aligned to input dates.
        Values for Jan 1–15 and Dec 17–31 are NaN (insufficient window data).
    """
    dates = pd.to_datetime(dates)
    
    # Pivot to wide format: years × calendar days
    df = pd.DataFrame({
        'year': dates.dt.year,
        'day': dates.dt.strftime('%m-%d'),
        'Tmax': tmax_series
    })
    tmax_wide = df.pivot(index='year', columns='day', values='Tmax')
    tmax_wide = tmax_wide.reindex(sorted(tmax_wide.columns), axis=1)
    tmax_values = tmax_wide.to_numpy()
    n_days = tmax_values.shape[1]
    
    # Compute 95th percentile for each calendar day (with 31-day window)
    p95_values = []
    for j in range(15, n_days - 15):
        window = np.concatenate([
            tmax_values[:, j-15:j].flatten(),
            tmax_values[:, j].flatten(),
            tmax_values[:, j+1:j+16].flatten()
        ])
        window = window[~np.isnan(window)]
        p95_values.append(np.quantile(window, 0.95) if len(window) > 0 else np.nan)
    
    # Map back to original date sequence (only valid for day-of-year 16–350)
    result = np.full(len(dates), np.nan)
    valid_mask = (dates.dt.dayofyear >= 16) & (dates.dt.dayofyear <= 350)
    result[valid_mask] = np.tile(p95_values, tmax_values.shape[0])[:valid_mask.sum()]
    return result


def process_grid_cell_p95(
    tmax_timeseries: np.ndarray,
    dates: pd.DatetimeIndex
) -> np.ndarray:
    """
    Compute day-of-year 95th percentile thresholds for a single grid cell.
    
    Parameters
    ----------
    tmax_timeseries : np.ndarray
        1D array of daily Tmax values for one grid cell (time dimension only)
    dates : pd.DatetimeIndex
        Dates corresponding to each timestep
    
    Returns
    -------
    np.ndarray
        366-element array of 95th percentile thresholds indexed by day-of-year
        (position 0 = Jan 1, position 365 = Dec 31). Days with insufficient data
        (Jan 1–15, Dec 17–31) contain NaN.
    """
    if np.all(np.isnan(tmax_timeseries)):
        return np.full(366, np.nan)
    
    # Compute daily thresholds aligned to input dates
    daily_thresholds = calculate_tmax_p95(tmax_timeseries, dates)
    
    # Aggregate to day-of-year representation (mean across years for each DOY)
    p95_doy = np.full(366, np.nan)
    dayofyear = dates.dayofyear.values
    for doy in range(1, 367):
        mask = (dayofyear == doy)
        if np.any(mask):
            p95_doy[doy - 1] = np.nanmean(daily_thresholds[mask])
    return p95_doy